# Monash University
# FIT3152 - Data Analytics
## Assignment 1, Semester 1, 2024

**Student ID:** 42857193  
**Focus Country:** Croatia  
**Project:** Analysis of country-level predictors of pro-social behaviours to reduce the spread of COVID-19 during the early stages of the pandemic

**AI statement:** Generative AI was used in this assignment for translation from R to Python

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
STUDENT_ID = 42857193
np.random.seed(STUDENT_ID)

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
sns.set_palette('husl')

## Task 1: Descriptive Analysis and Pre-processing
### 1(a) - Load and Describe Data

In [ ]:
# Load data (from parent directory)
cvbase = pd.read_csv('../PsyCoronaBaselineExtract.csv')

# Sample 40,000 rows based on student ID
cvbase = cvbase.sample(n=40000, random_state=STUDENT_ID).reset_index(drop=True)

print(f"Dataset dimensions: {cvbase.shape[0]} rows × {cvbase.shape[1]} columns")

In [ ]:
# Data types
print("\nData types:")
print(cvbase.dtypes.value_counts())

print("\nText attributes:")
text_cols = cvbase.select_dtypes(include=['object']).columns.tolist()
print(text_cols)

In [ ]:
# Summary statistics
cvbase.describe()

In [ ]:
# Missing values analysis
missing = cvbase.isnull().sum()
missing_pct = (missing / len(cvbase)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})

print("Columns with missing values:")
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False))

In [ ]:
# Country analysis
country_counts = cvbase['coded_country'].value_counts()
print(f"Number of unique countries: {cvbase['coded_country'].nunique()}")
print(f"\nCountry with most responses: {country_counts.index[0]} ({country_counts.iloc[0]})")
print(f"Country with least responses: {country_counts.index[-1]} ({country_counts.iloc[-1]})")

if 'Croatia' in country_counts.index:
    print(f"\nCroatia responses: {country_counts['Croatia']}")

In [ ]:
# Age analysis
print(f"Mean age group: {cvbase['age'].mean():.3f}")
print("This indicates that the majority of participants are likely aged between 35-44 years.")

### 1(b) - Data Preprocessing

In [ ]:
# Replace missing values with 0
cvbase = cvbase.fillna(0)
print(f"Missing values after preprocessing: {cvbase.isnull().sum().sum()}")

## Task 2: Focus Country vs All Other Countries
### 2(a) - Compare Croatia to Other Countries

In [ ]:
# Split data
FOCUS_COUNTRY = 'Croatia'

croatia = cvbase[cvbase['coded_country'] == FOCUS_COUNTRY].copy()
others = cvbase[cvbase['coded_country'] != FOCUS_COUNTRY].copy()

print(f"{FOCUS_COUNTRY} data: {len(croatia)} rows")
print(f"Other countries data: {len(others)} rows")

In [ ]:
# Calculate means for Croatia
numeric_cols = croatia.select_dtypes(include=[np.number]).columns.tolist()
croatia_means = croatia[numeric_cols].mean().sort_values()

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
croatia_means.plot(kind='barh', ax=ax, color='purple')
ax.set_xlabel('Mean responses')
ax.set_ylabel('Survey questions')
ax.set_title('Mean of responses for each question in Croatia')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate means for other countries
others_means = others[numeric_cols].mean().sort_values()

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
others_means.plot(kind='barh', ax=ax, color='lightblue')
ax.set_xlabel('Mean of responses')
ax.set_ylabel('Survey questions')
ax.set_title('Mean of responses for each question over the globe')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### 2(b) - Predict Pro-social Attitudes for Croatia

In [ ]:
# Correlation matrix for Croatia
numeric_croatia = croatia.select_dtypes(include=[np.number])
croatia_corr = numeric_croatia.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    croatia_corr,
    cmap='RdBu_r',
    center=0,
    vmin=-0.5,
    vmax=1.0,
    square=True,
    ax=ax,
    cbar_kws={'label': 'correlation'}
)
ax.set_title("Correlation between each of Croatia's predictors")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Function to fit linear models and report results
def model_evaluated(X, y, target_name):
    """Fit linear regression and report statistics."""
    # Remove missing values
    mask = ~(X.isnull().any(axis=1) | y.isnull())
    X_clean = X[mask]
    y_clean = y[mask]
    
    if len(X_clean) == 0:
        return None
    
    # Fit model
    model = LinearRegression()
    model.fit(X_clean, y_clean)
    
    # R-squared
    r_squared = model.score(X_clean, y_clean)
    
    # Adjusted R-squared
    n = len(y_clean)
    p = X_clean.shape[1]
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - p - 1)
    
    # Calculate p-values (handle singular matrix with pseudo-inverse)
    predictions = model.predict(X_clean)
    residuals = y_clean - predictions
    mse = np.sum(residuals**2) / (n - p - 1)
    
    try:
        # Try regular inverse
        XTX_inv = np.linalg.inv(X_clean.T.dot(X_clean))
        var_coef = mse * XTX_inv.diagonal()
    except np.linalg.LinAlgError:
        # Use Moore-Penrose pseudo-inverse if matrix is singular
        XTX_pinv = np.linalg.pinv(X_clean.T.dot(X_clean))
        var_coef = mse * XTX_pinv.diagonal()
    
    std_errors = np.sqrt(np.abs(var_coef))  # abs to handle numerical errors
    
    # T-statistics (avoid division by zero)
    t_stats = np.divide(model.coef_, std_errors,
                        out=np.zeros_like(model.coef_),
                        where=std_errors!=0)
    
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), n - p - 1))
    
    # Significant predictors (p < 0.001)
    significant_mask = p_values < 0.001
    significant_features = X_clean.columns[significant_mask].tolist()
    significant_coefs = model.coef_[significant_mask]
    
    print(f"\n{target_name}")
    print(f"R-squared: {r_squared:.6f}")
    print(f"Adjusted R-squared: {adj_r_squared:.6f}")
    print("99.9% confidence interval significant predictors:")
    if len(significant_features) > 0:
        for feat, coef in zip(significant_features, significant_coefs):
            print(f"  {feat}: {coef:.6f}")
    else:
        print("  None")
    
    return {
        'r_squared': r_squared,
        'adj_r_squared': adj_r_squared,
        'significant_features': significant_features,
        'significant_coefs': significant_coefs
    }

# Fit models for Croatia
prosocial_vars = ['c19ProSo01', 'c19ProSo02', 'c19ProSo03', 'c19ProSo04']
exclude_cols = ['coded_country']

croatia_results = {}
print("Pro-social attitudes in Croatia predictors model summary")
print("="*60)

for target in prosocial_vars:
    feature_cols = [col for col in croatia.columns
                   if col not in exclude_cols + prosocial_vars]
    X = croatia[feature_cols].select_dtypes(include=[np.number])
    y = croatia[target]
    
    result = model_evaluated(X, y, target)
    croatia_results[target] = result

### 2(c) - Predict Pro-social Attitudes for Rest of World

In [ ]:
# Correlation matrix for rest of world
numeric_others = others.select_dtypes(include=[np.number])
global_corr = numeric_others.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    global_corr,
    cmap='RdBu_r',
    center=0,
    vmin=-0.5,
    vmax=1.0,
    square=True,
    ax=ax,
    cbar_kws={'label': 'correlation'}
)
ax.set_title("Correlation between each of the world's predictors")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Fit models for rest of world
others_results = {}
print("Pro-social attitudes in the world predictors model summary")
print("="*60)

for target in prosocial_vars:
    feature_cols = [col for col in others.columns
                   if col not in exclude_cols + prosocial_vars]
    X = others[feature_cols].select_dtypes(include=[np.number])
    y = others[target]
    
    result = model_evaluated(X, y, target)
    others_results[target] = result

## Task 3: Focus Country vs Cluster of Similar Countries
### 3(a) - K-means Clustering

In [ ]:
# Load external clustering data (from parent directory)
external = pd.read_csv('../task3.csv')
print(f"External data shape: {external.shape}")
external.head()

In [ ]:
# Remove countries with NA values
cleaned_external = external.dropna()
print(f"Countries after removing NAs: {len(cleaned_external)}")

# Scale numeric columns (excluding country name)
numeric_cols = cleaned_external.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
scaled_data = scaler.fit_transform(cleaned_external[numeric_cols])

# Perform K-means clustering
n_clusters = round(len(cleaned_external) / 5)
print(f"Number of clusters: {n_clusters}")

kmeans = KMeans(n_clusters=n_clusters, n_init=15, random_state=STUDENT_ID)
clusters = kmeans.fit_predict(scaled_data)

# Create clusters dataframe
clusters_df = pd.DataFrame({
    'country': cleaned_external['country'].values,
    'cluster': clusters
})

# Find Croatia's cluster
croatia_cluster = clusters_df[clusters_df['country'] == 'Croatia']['cluster'].values[0]
similar = clusters_df[clusters_df['cluster'] == croatia_cluster]

print(f"\nCroatia is in cluster {croatia_cluster}")
print("\nSimilar countries:")
print(similar)

### 3(b) - Analyze Similar Countries Cluster

In [ ]:
# Extract baseline data for similar countries (excluding Croatia)
similar_country_list = similar['country'].tolist()
clustered = cvbase[cvbase['coded_country'].isin(similar_country_list)].copy()
clustered = clustered[clustered['coded_country'] != 'Croatia']

print(f"Similar countries data (excl. Croatia): {len(clustered)} rows")
print(f"Countries: {clustered['coded_country'].unique()}")

In [ ]:
# Correlation matrix for similar countries
numeric_clustered = clustered.select_dtypes(include=[np.number])
clustered_corr = numeric_clustered.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    clustered_corr,
    cmap='RdBu_r',
    center=0,
    vmin=-0.5,
    vmax=1.0,
    square=True,
    ax=ax,
    cbar_kws={'label': 'correlation'}
)
ax.set_title('Correlation between predictors for countries similar to Croatia')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Fit models for similar countries
similar_results = {}
print("Model Summary for countries similar to Croatia to predict pro-social attitudes")
print("="*60)

for target in prosocial_vars:
    feature_cols = [col for col in clustered.columns
                   if col not in exclude_cols + prosocial_vars]
    X = clustered[feature_cols].select_dtypes(include=[np.number])
    y = clustered[target]
    
    result = model_evaluated(X, y, target)
    similar_results[target] = result

## Comparison of Significant Predictors Across All Models

In [ ]:
# Create predictor comparison table
all_predictors = set()
all_models = []

# Collect predictors from all models
for target, results in croatia_results.items():
    if results:
        model_name = f"Croatia_{target}"
        all_models.append(model_name)
        all_predictors.update(results['significant_features'])

for target, results in others_results.items():
    if results:
        model_name = f"RoW_{target}"
        all_models.append(model_name)
        all_predictors.update(results['significant_features'])

for target, results in similar_results.items():
    if results:
        model_name = f"Similar_{target}"
        all_models.append(model_name)
        all_predictors.update(results['significant_features'])

# Create table
predictor_table = pd.DataFrame(0, index=sorted(all_predictors), columns=all_models)

# Fill table
for target, results in croatia_results.items():
    if results:
        model_name = f"Croatia_{target}"
        for feat in results['significant_features']:
            predictor_table.loc[feat, model_name] = 1

for target, results in others_results.items():
    if results:
        model_name = f"RoW_{target}"
        for feat in results['significant_features']:
            predictor_table.loc[feat, model_name] = 1

for target, results in similar_results.items():
    if results:
        model_name = f"Similar_{target}"
        for feat in results['significant_features']:
            predictor_table.loc[feat, model_name] = 1

# Visualize
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    predictor_table,
    cmap=['lightgray', 'green'],
    cbar=False,
    linewidths=0.5,
    linecolor='black',
    square=True,
    ax=ax
)
ax.set_title('Significant predictors for individual models')
ax.set_xlabel('Models')
ax.set_ylabel('Predictors')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Summary and Conclusions

This analysis has translated the R-based Assignment 1 to Python, implementing:

1. **Descriptive Analysis**: Analyzed the PsyCorona dataset structure, missing values, and key statistics
2. **Country Comparison**: Compared Croatia's responses to the rest of the world
3. **Predictive Modeling**: Fitted linear regression models to predict pro-social attitudes
4. **Clustering**: Identified similar countries to Croatia using K-means clustering
5. **Comparative Analysis**: Examined predictor patterns across different country groupings

The Python implementation uses:
- `pandas` for data manipulation
- `numpy` for numerical operations
- `matplotlib` and `seaborn` for visualization
- `scikit-learn` for machine learning (clustering and regression)
- `scipy` for statistical tests